In [0]:
%sql

-- 1. Create Schemas
CREATE SCHEMA IF NOT EXISTS main.reference_data
COMMENT 'Static lookup tables like enums';

CREATE SCHEMA IF NOT EXISTS main.ordersystem_sample
COMMENT 'Sample / fake data tables for development and testing';

In [0]:
%sql

DROP TABLE IF EXISTS main.ordersystem_sample.order_line_items;
DROP TABLE IF EXISTS main.ordersystem_sample.orders;
DROP TABLE IF EXISTS main.ordersystem_sample.products;
DROP TABLE IF EXISTS main.ordersystem_sample.stores;
DROP TABLE IF EXISTS main.ordersystem_sample.customers;
DROP TABLE IF EXISTS main.reference_data.order_status_enum;
DROP TABLE IF EXISTS main.reference_data.item_status_enum;

In [0]:
%sql

-- -------------------------------------------------------------------
-- 2. Create Enums (Reference Tables)
-- -------------------------------------------------------------------

CREATE TABLE IF NOT EXISTS main.reference_data.order_status_enum (
  status STRING COMMENT 'Valid values for order status'
);

INSERT INTO main.reference_data.order_status_enum (status) VALUES
  ('PENDING'),
  ('SHIPPED'),
  ('DELIVERED'),
  ('CANCELLED'),
  ('COMPLETED');

CREATE TABLE IF NOT EXISTS main.reference_data.item_status_enum (
  status STRING COMMENT 'Valid values for order line item status'
);

INSERT INTO main.reference_data.item_status_enum (status) VALUES
  ('PENDING'),
  ('ALLOCATED'),
  ('SHIPPED'),
  ('DELIVERED'),
  ('CANCELLED');

In [0]:
%sql
-- Customers
CREATE TABLE IF NOT EXISTS main.ordersystem_sample.customers (
  customer_id STRING PRIMARY KEY,
  name STRING NOT NULL,
  email STRING,
  phone STRING,
  address_json STRING, -- JSON-encoded address
  is_deleted BOOLEAN,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
) COMMENT 'Customers with soft deletes and JSON address field';

-- Stores
CREATE TABLE IF NOT EXISTS main.ordersystem_sample.stores (
  store_id STRING PRIMARY KEY,
  store_name STRING NOT NULL,
  location STRING
) COMMENT 'Stores fulfilling orders';

-- Products
CREATE TABLE IF NOT EXISTS main.ordersystem_sample.products (
  product_id STRING PRIMARY KEY,
  product_name STRING NOT NULL,
  price DECIMAL(10,2)
) COMMENT 'Products available for ordering';

-- Orders
CREATE TABLE IF NOT EXISTS main.ordersystem_sample.orders (
  order_id STRING PRIMARY KEY,
  customer_id STRING NOT NULL,
  store_id STRING NOT NULL,
  order_status STRING NOT NULL,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES main.ordersystem_sample.customers(customer_id),
  CONSTRAINT fk_orders_store FOREIGN KEY (store_id) REFERENCES main.ordersystem_sample.stores(store_id)
) COMMENT 'Orders placed by customers';

-- Order Line Items
CREATE TABLE IF NOT EXISTS main.ordersystem_sample.order_line_items (
  line_item_id STRING PRIMARY KEY,
  order_id STRING NOT NULL,
  product_id STRING NOT NULL,
  quantity INT NOT NULL,
  item_status STRING NOT NULL,
  created_at TIMESTAMP,
  CONSTRAINT fk_lineitems_order FOREIGN KEY (order_id) REFERENCES main.ordersystem_sample.orders(order_id),
  CONSTRAINT fk_lineitems_product FOREIGN KEY (product_id) REFERENCES main.ordersystem_sample.products(product_id)
) COMMENT 'Individual line items associated with an order';